# Laboratorio 7 — Simulación Monte Carlo: Álbum de Figuritas
**MM3014 · Teoría de Probabilidades · UVG**

## Introducción

En este laboratorio simulamos el proceso de completar un álbum de **N = 100** figuritas distintas comprando sobres de **S = 7** figuritas únicas cada uno (sin repetición dentro del mismo sobre). Se realizan **R = 10,000** simulaciones independientes para estimar:

- La cantidad media de sobres necesarios para completar el álbum.
- La cantidad de figuritas repetidas generadas en el proceso.
- La probabilidad de necesitar más de 30 sobres.

Los resultados se comparan con la predicción teórica basada en el **problema del coleccionista de cupones** (*coupon collector's problem*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

np.random.seed(2026)

In [ ]:
N = 100    # total de figuritas distintas
S = 7      # figuritas por sobre (únicas dentro del sobre)
R = 10000  # número de simulaciones

In [ ]:
packs_results    = []
repeated_results = []

for _ in range(R):
    collected         = np.zeros(N, dtype=bool)
    packs_bought      = 0
    repeated_stickers = 0

    while not collected.all():
        pack = np.random.choice(N, S, replace=False)
        for sticker in pack:
            if collected[sticker]:
                repeated_stickers += 1
            else:
                collected[sticker] = True
        packs_bought += 1

    packs_results.append(packs_bought)
    repeated_results.append(repeated_stickers)

packs_results    = np.array(packs_results)
repeated_results = np.array(repeated_results)

In [ ]:
mean_packs    = np.mean(packs_results)
std_packs     = np.std(packs_results)
mean_repeated = np.mean(repeated_results)
std_repeated  = np.std(repeated_results)
p_over_30     = np.mean(packs_results > 30)
theo_min      = math.ceil(N / S)  # = 15

print("=" * 55)
print("RESULTADOS DE LA SIMULACIÓN")
print("=" * 55)
print(f"Media de sobres comprados:          {mean_packs:.4f}")
print(f"Desviación estándar de sobres:      {std_packs:.4f}")
print(f"Media de figuritas repetidas:       {mean_repeated:.4f}")
print(f"Desviación estándar de repetidas:   {std_repeated:.4f}")
print(f"P(sobres > 30):                     {p_over_30:.4f}")
print()
print("Justificación del umbral de 30 sobres:")
print(f"  Mínimo teórico: ceil({N}/{S}) = {theo_min} sobres.")
print(f"  30 es el doble del mínimo ({theo_min}), umbral razonable")
print(f"  para cuantificar la cola de la distribución.")
print(f"  P(sobres > 30) = {p_over_30:.4f} refleja la frecuencia")
print(f"  de simulaciones que superan ese doble del mínimo.")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(packs_results, bins=40, edgecolor='black', color='steelblue', alpha=0.7)
plt.axvline(mean_packs, color='red',   linestyle='--', linewidth=2,
            label=f'Media muestral = {mean_packs:.2f}')
plt.axvline(theo_min,   color='green', linestyle='-',  linewidth=2,
            label=f'Mínimo teórico = {theo_min}')
plt.title('Distribución del número de sobres para completar el álbum\n'
          f'(N={N}, S={S}, R={R:,} simulaciones)')
plt.xlabel('Número de sobres comprados')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.savefig('distribucion_sobres.png', dpi=150)
plt.show()

## Pregunta 1: Mínimo de sobres sin figuritas repetidas

Si ninguna figurita se repitiera jamás, cada sobre aportaría exactamente $S$ figuritas completamente nuevas. En ese caso ideal:

$$\text{Mínimo de sobres} = \left\lceil \frac{N}{S} \right\rceil = \left\lceil \frac{100}{7} \right\rceil = \lceil 14.2857 \rceil = 15$$

Este es el **límite inferior absoluto**: con 14 sobres solo se abren $14 \times 7 = 98$ figuritas, insuficiente para completar el álbum de 100. Con 15 sobres se abren $15 \times 7 = 105$ figuritas, lo que sería suficiente **solo si no hubiera ninguna repetición**.

¿Ocurre este caso en las simulaciones? Lo verificamos a continuación.

In [ ]:
min_count = int(np.sum(packs_results == theo_min))
print(f"Mínimo teórico: ceil({N}/{S}) = {theo_min} sobres")
print(f"Cálculo: {N} / {S} = {N/S:.6f} → redondeando arriba = {theo_min}")
print()
print(f"Simulaciones que terminaron en exactamente {theo_min} sobres: {min_count} de {R}")
print(f"Proporción: {100*min_count/R:.4f}%")
print()
if min_count == 0:
    print("No se observó este caso en ninguna de las 10,000 simulaciones.")
    print("La probabilidad de que ocurra es extremadamente pequeña:")
    print("requeriría que las 15 combinaciones de 7 figuritas cubran")
    print("exactamente las 100 sin ninguna repetición, lo cual es casi imposible.")
else:
    print(f"Se observó {min_count} vez/veces — evento extremadamente raro.")

## Pregunta 2: Predicción teórica con el número armónico $H_{100}$

El **problema del coleccionista de cupones** nos da la esperanza del número de sobres necesarios. Para packs de $S$ figuritas únicas, la fórmula es:

$$E[T] = \frac{N}{S} \cdot H_N$$

donde $H_N$ es el $N$-ésimo número harmónico, aproximado por:

$$H_N \approx \ln(N) + \gamma \quad (\gamma = 0.5772\text{ — constante de Euler-Mascheroni})$$

Aplicando para $N = 100$, $S = 7$:

In [ ]:
H_N = np.log(N) + 0.5772
E_T = (N / S) * H_N

print(f"H_100 ≈ ln(100) + 0.5772")
print(f"      = {np.log(N):.4f} + 0.5772")
print(f"      = {H_N:.4f}")
print()
print(f"E[T] = (N/S) × H_100")
print(f"     = ({N}/{S}) × {H_N:.4f}")
print(f"     = {N/S:.4f} × {H_N:.4f}")
print(f"     = {E_T:.4f} sobres")
print()
print(f"Media simulada:    {mean_packs:.4f} sobres")
diff = abs(E_T - mean_packs)
diff_pct = 100 * diff / mean_packs
print(f"Diferencia:        {diff:.4f} sobres  ({diff_pct:.2f}%)")
print()
print("La aproximación teórica es muy cercana a la media simulada,")
print("lo que valida tanto el modelo matemático como la simulación.")

## Pregunta 3: Número esperado de figuritas repetidas

El total de figuritas abiertas al completar el álbum es $\text{sobres comprados} \times S$. De ese total, exactamente $N$ son figuritas **nuevas** (las que completan el álbum). El resto son repetidas:

$$E[\text{repetidas}] = E[\text{sobres}] \cdot S - N = E[T] \cdot S - N$$

Usando la estimación teórica $E[T]$ calculada en la Pregunta 2:

In [ ]:
E_repeated = E_T * S - N

print(f"Total de figuritas abiertas ≈ E[T] × S = {E_T:.4f} × {S} = {E_T*S:.4f}")
print(f"Figuritas nuevas necesarias  = N = {N}")
print(f"E[repetidas] = {E_T*S:.4f} - {N} = {E_repeated:.4f}")
print()
print(f"Media simulada de repetidas: {mean_repeated:.4f}")
print(f"Diferencia absoluta:         {abs(E_repeated - mean_repeated):.4f} figuritas")
print(f"Diferencia relativa:         {100*abs(E_repeated-mean_repeated)/mean_repeated:.2f}%")
print()
print("El modelo teórico estima bien la cantidad de repetidas,")
print("confirmando que el proceso de colección genera un número")
print("significativo de duplicados (muchos más que las N=100 necesarias).")

## Pregunta 4: Interpretación de la desviación estándar

La desviación estándar del número de sobres necesarios es **alta en relación con la media**. Para cuantificar esto usamos el **coeficiente de variación** (CV = $\sigma / \mu$):

$$\text{CV} = \frac{\sigma}{\mu} \approx \frac{\text{std\_packs}}{\text{mean\_packs}}$$

Un CV superior al 20–25% indica alta variabilidad relativa. En este problema, el CV es elevado debido a la **naturaleza asimétrica del proceso de colección**:

1. **Fase inicial (fácil):** al principio casi cualquier figurita que sale es nueva, por lo que el álbum avanza rápidamente.
2. **Fase final (difícil — efecto de cola larga):** cuando faltan pocas figuritas, la probabilidad de obtener una nueva en cada sobre es muy baja. Cada figurita faltante se vuelve exponencialmente más costosa de conseguir.
3. **Alta varianza por azar puro:** no hay ningún control sobre el contenido de los sobres. Algunos coleccionistas tienen suerte y terminan cerca del mínimo; otros deben comprar el doble o triple de sobres. Esta dispersión se captura en la desviación estándar.

El histograma anterior ilustra esta asimetría positiva: la distribución tiene una cola derecha larga, lo que significa que aunque la mayoría termina cerca de la media, una fracción significativa de simulaciones requiere muchos más sobres de lo esperado. Esto es intrínseco al problema del coleccionista y no puede eliminarse sin mecanismos de control (como intercambio de figuritas).